The distinction is standard in learning analytics literature:

Background models answer:

“Who is statistically at risk before the course begins?”

Behavioral / temporal models answer:

“Who is becoming at risk based on how they engage and perform?”

_____

They are sufficient to separate “at risk” vs “not at risk”, but not sufficient to reliably separate 4 fine-grained outcomes.

This is documented in OULAD studies.

Background features are predictive of risk, not grade quality.

📌 Multiclass increases label noise without adding information.

____

Scientifically standard mapping:

Fail + Withdraw → 1 (at risk)-----> not likely to be accepted

Pass + Distinction → 0 (not at risk)-----> likely to be accepted

This mapping is widely used in literature.

____

We focus on binary background risk prediction because studentInfo contains static pre-enrollment variables. These are suitable for identifying at-risk students, but insufficient for fine-grained outcome differentiation. This aligns with the dataset’s original design and prior learning analytics literature.

_____

A single student may:

take different modules

retake the same module in a later presentation

withdraw from one module and pass another

____

Important clarifications (this fixes common confusion)

It is module-specific, not global

It is known at enrollment time

It does not count:

attempts in other modules

partial submissions

general academic failures

# During EDA

nulls and how it is going to be handled and the inconsistant input in the imd_band

categorical data

duplicates

scalling based on the model then  modeling



In [7]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, classification_report

In [8]:
data= pd.read_csv("/content/studentInfo.csv")

In [9]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32593 entries, 0 to 32592
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   code_module           32593 non-null  object
 1   code_presentation     32593 non-null  object
 2   id_student            32593 non-null  int64 
 3   gender                32593 non-null  object
 4   region                32593 non-null  object
 5   highest_education     32593 non-null  object
 6   imd_band              31482 non-null  object
 7   age_band              32593 non-null  object
 8   num_of_prev_attempts  32593 non-null  int64 
 9   studied_credits       32593 non-null  int64 
 10  disability            32593 non-null  object
 11  final_result          32593 non-null  object
dtypes: int64(3), object(9)
memory usage: 3.0+ MB


In [10]:
data['final_result'].unique()

array(['Pass', 'Withdrawn', 'Fail', 'Distinction'], dtype=object)

In [11]:
data.isnull().sum()

,0
code_module,0
code_presentation,0
id_student,0
gender,0
region,0
highest_education,0
imd_band,1111
age_band,0
num_of_prev_attempts,0
studied_credits,0


In [12]:
data['id_student'].nunique()

28785

In [13]:
data.duplicated().sum()

np.int64(0)

In [14]:
# Replace invalid IMD value with NaN
data["imd_band"] = data["imd_band"].replace("20-Oct", np.nan)


In [15]:
# target mapping
data["target"] = data["final_result"].map({
    "Fail": 1,
    "Withdrawn": 1,
    "Pass": 0,
    "Distinction": 0
})


In [16]:
data = data.drop(columns=["final_result"])


In [17]:
data['target'].value_counts()

,count
target,
1,17208
0,15385


In [18]:
X = data.drop(columns=["target"])
y = data["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)


In [19]:
numerical = [
    "studied_credits",
    "num_of_prev_attempts"
]

ordinal = [
    "age_band",
    "highest_education",
    "imd_band"
]

nominal= [
    "gender",
    "region",
    "disability"
]


In [20]:
age_order = [
    "0-35", "35-55", "55<="
]

education_order = [
    "No Formal quals",
    "Lower Than A Level",
    "A Level or Equivalent",
    "HE Qualification",
    "Post Graduate Qualification"
]

imd_order = [
    "0-10%",
    "10-20%",
    "20-30%",
    "30-40%",
    "40-50%",
    "50-60%",
    "60-70%",
    "70-80%",
    "80-90%",
    "90-100%"
]


***Preprocessing helping functions***

In [21]:
num_pipeline = Pipeline([
    # ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])


In [22]:
ord_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
    ("encoder", OrdinalEncoder(
        categories=[age_order, education_order, imd_order + ["Missing"]],
        handle_unknown="use_encoded_value",
        unknown_value=-1
    ))
])


In [23]:
nom_pipeline = Pipeline([
    # ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])


In [24]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", num_pipeline, numerical),
        ("ord", ord_pipeline, ordinal),
        ("nom", nom_pipeline, nominal)
    ]
)


In [25]:
X_train_pre = preprocessor.fit_transform(X_train)

X_test_pre= preprocessor.transform(X_test)


In [26]:
models = {
    "XGB": XGBClassifier(
        eval_metric="logloss",
        random_state=42
    )
}


'LogReg': np.float64(0.6494766705498255)

 'RF': np.float64(0.5918067584335958)

 'XGB': np.float64(0.6484666652944369)

In [28]:
results = {}

for name, model in models.items():
    pipe = Pipeline([
        ("prep", preprocessor),
        ("model", model) # Add the model to the pipeline
    ])

    pipe.fit(X_train, y_train)
    preds = pipe.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, preds)
    results[name] = auc

print(results)

{'XGB': np.float64(0.6484666652944369)}


In [29]:
print(classification_report(y_test, pipe.predict(X_test)))

              precision    recall  f1-score   support

           0       0.59      0.57      0.58      3077
           1       0.63      0.65      0.64      3442

    accuracy                           0.61      6519
   macro avg       0.61      0.61      0.61      6519
weighted avg       0.61      0.61      0.61      6519

